# SaralGati - Custom LoRA Fine-Tuning with Unsloth

This notebook fine-tunes **Meta's Llama-3.2-3B-Instruct** using Unsloth (for 2x faster training on the free T4 GPU) to create a custom LoRA adapter for guiding Indian elders on mobile apps.

In [ ]:
%%capture
# Install Unsloth and Xformers
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" trl peft accelerate bitsandbytes
!pip install datasets

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Can increase later if needed
dtype = None # Auto detects for Float16/Bfloat16
load_in_4bit = True # Use 4bit quantization to fit on T4 GPU

# Load Llama-3.2-3B-Instruct base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Dropout = 0 is optimized
    bias = "none",    # Bias = "none" is optimized
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
from datasets import load_dataset
import os
from google.colab import files
from unsloth.chat_templates import get_chat_template

# 1. Setup Llama 3.2 / 3.1 Chat Template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

# 2. Upload saralgati_multiturn_train.jsonl to Colab
print("Please upload saralgati_multiturn_train.jsonl dataset file:")
uploaded = files.upload()

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="saralgati_multiturn_train.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Mask loss on user/system prompts so model focuses 100% on learning proper elder responses & TARGET tags
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer_stats = trainer.train()

In [ ]:
# Save the LoRA adapter locally
model.save_pretrained("saralgati_lora_model")
tokenizer.save_pretrained("saralgati_lora_model")

import shutil
shutil.make_archive("saralgati_lora", 'zip', "saralgati_lora_model")
print("\n✅ Training complete! Adapter saved as saralgati_lora.zip")
files.download("saralgati_lora.zip")